## intro

### openai sdk with open router

In [2]:
from os import getenv

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # loads .env into environment variables


def main() -> None:
    print("Hello from start-work-with-agent!")

    api_key = getenv("OPENROUTER_API_KEY")
    if not api_key:
        raise SystemExit("OPENROUTER_API_KEY not found. Check your .env file.")

    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=api_key,
    )
    
    # creat request and get response 
    
    response = client.responses.create(
        model="openrouter/free",
        input="Say this is a test",
    )
    
    print(response.output_text)

### open router sdk

#### What `with` does

`with` is Python's way of saying: "set something up, let me use it, and **always clean it up afterward**, even if an error happens."

**The problem it solves**

Some objects hold resources that must be released: files, network connections, database connections, locks. Without `with`, you have to close them manually:

```python
client = OpenRouter(api_key=api_key)
try:
    res = client.chat.send(...)
finally:
    client.close()  # must run even if send() fails
```

If you forget the `finally`, an error in `send()` skips `close()` and the connection stays open (a resource leak). `with` does this for you.

**Same code using `with`**

```python
with OpenRouter(api_key=api_key) as client:
    res = client.chat.send(...)
# here, the client is already closed
```

This is equivalent to the `try/finally` version, but shorter and harder to get wrong.

**How it works internally**

Any object that works with `with` is a **context manager**. It defines two special methods:

- `__enter__()` runs when the block starts. Its return value is what `as client` receives.
- `__exit__()` runs when the block ends, whether normally or because of an exception. This is where cleanup happens.

So the flow is:

1. `OpenRouter(api_key=api_key)` creates the object.
2. Python calls `__enter__()` and assigns the result to `client`.
3. Your indented code runs.
4. Python calls `__exit__()` to close and clean up, no matter what.

**The `as client` part**

`as client` just gives a name to the object returned by `__enter__()`. Usually that is the object itself, but not always. Compare with a file:

```python
with open("notes.txt") as f:
    text = f.read()
# the file is automatically closed here
```

Here `f` is the opened file, and Python closes it for you when the block ends.

**Why it matters**

- **Safety:** cleanup runs even if the code inside raises an exception.
- **Readability:** setup and cleanup are visible in one place, and the indentation shows exactly where the resource is in use.
- **Fewer leaks:** you can't forget to close it.
**Important detail**

After the `with` block ends, `client` is closed and you shouldn't use it anymore. That's why the request is made inside the block. The response `res` is just data, so it is still safe to use afterward, which is what the fixed code does with `print(res.choices[0].message.content)`.

**Making your own**

You can write a context manager yourself:

```python
class Demo:
    def __enter__(self):
        print("setup")
        return self

    def __exit__(self, exc_type, exc, tb):
        print("cleanup")

with Demo() as d:
    print("working")
```

Output:
```
setup
working
cleanup
```

Even if you raise an error inside the block, `cleanup` is still printed.

#### example for use open router sdk

In [3]:
from os import getenv

from dotenv import load_dotenv
load_dotenv()  # loads .env into environment variables
from openrouter import OpenRouter



def main() -> None:
    api_key = getenv("OPENROUTER_API_KEY")
    if not api_key:
        raise SystemExit("OPENROUTER_API_KEY not found. Check your .env file.")

    with OpenRouter(api_key=api_key) as client:
        res = client.chat.send(
            model="openrouter/free",
            messages=[
                {"role": "user", "content": "What is the capital of France?"},
            ],
            stream=False,
        )

        message = [{"role":"user","content":"tell me fun fact"}]
        res2 = client.chat.send(
            model="openrouter/free",
            messages=message,
            stream=False,
        )

    # Print only the model's answer, not the whole response object
    print(res.choices[0].message.content)
    print(res2.choices[0].message.content)




#### broken code and fix 

In [4]:
# broken code 

from os import getenv

from dotenv import load_dotenv
load_dotenv()  # loads .env into environment variables
from openrouter import OpenRouter



def main() -> None:
    api_key = getenv("OPENROUTER_API_KEY")
    if not api_key:
        raise SystemExit("OPENROUTER_API_KEY not found. Check your .env file.")

    with OpenRouter(api_key=api_key) as client:
        res = client.chat.send(
            model="openrouter/free",
            messages=[
                {"role": "user", "content": "What is the capital of France?"},
            ],
            stream=False,
        )

        message = [{"role":"user","content":"tell me fun fact"}]
        res2 = client.chat.send(
            model="openrouter/free",
            messages=message,
            stream=False,
        )

    # Print only the model's answer, not the whole response object
    print(res.choices[0].message.content)
    print(res2.choices[0].message.content)
    # another example
    with OpenRouter(api_key=api_key) as client:
        res3 = client.chat.send(
            model="openrouter/free",
            messages=[
                {"role": "user", "content": "tell me info about egypt"},
            ],
            stream=False,
        )
    print(res3.choices[0].message.content)

In [5]:
# fix
from os import getenv

from dotenv import load_dotenv
from openrouter import OpenRouter

load_dotenv()


def ask(client: OpenRouter, question: str) -> str:
    res = client.chat.send(
        model="openrouter/free",
        messages=[{"role": "user", "content": question}],
        stream=False,
        timeout_ms=60_000,  # give up after 60 seconds
    )
    return res.choices[0].message.content


def main() -> None:
    api_key = getenv("OPENROUTER_API_KEY")
    if not api_key:
        raise SystemExit("OPENROUTER_API_KEY not found. Check your .env file.")

    questions = [
        "What is the capital of France?",
        "tell me fun fact",
        "tell me info about egypt",
    ]

    with OpenRouter(api_key=api_key) as client:
        for q in questions:
            print(f"\n>>> {q}")
            try:
                print(ask(client, q))
            except Exception as e:
                print(f"Request failed: {e}")



#### in jupyter Notebook

In [ ]:
# just run in jupyter notebook

from os import getenv

from dotenv import load_dotenv
load_dotenv()  # loads .env into environment variables
from openrouter import OpenRouter

from IPython.display import Markdown,display

def main() -> None:
    api_key = getenv("OPENROUTER_API_KEY")
    if not api_key:
        raise SystemExit("OPENROUTER_API_KEY not found. Check your .env file.")

    with OpenRouter(api_key=api_key) as client:
        res = client.chat.send(
            model="openrouter/free",
            messages=[
                {"role": "user", "content": "tell me about egypt"},
            ],
            stream=False,
        )
    # Print only the model's answer, not the whole response object
    answer = res.choices[0].message.content
    print(answer)
    # 
    display(Markdown(answer))
    
    #! we can even take response and give it to next model and work with it inside messege 
    #! and this call it chain multiples calls 
    #TODO: add here code example to do it 
    # 
    
main();

## Agent

### Basics Knowledge

**3 definitions of an AI Agent**

**Sam Altman**
"AI systems that can do work for you independently"

**Early 2025 consensus (Anthropic, Hugging Face, ..)**
AI systems where an LLM controls the workflow

**Current prevailing view**
An LLM with Tools in a Loop to achieve a goal

---

**Anthropic distinguish two types:**

1. **Workflows are systems where LLMs and tools are orchestrated through predefined code paths**

2. **Agents are dynamic systems where LLMs direct their own processes and tool usage, maintaining control over how they accomplish tasks**

---

**Part 2: Agentic Systems (Anthropic's Distinction)**

Anthropic distinguishes between two types of agentic systems:

*   **Workflows:** Systems where LLMs and tools are orchestrated through **predefined code paths**.
*   **Agents:** Dynamic systems where LLMs **direct their own processes** and tool usage, maintaining control over how they accomplish tasks.

**By contrast, Agents are:**

1.  **Open-ended:** They can handle tasks they weren't specifically trained for.
2.  **Feedback loops:** They take actions, observe the results, and adjust.
3.  **No fixed path:** They decide the steps to take on the fly.

> **Simple Example:** 
> *   **Workflow:** A factory assembly line. Every car goes through the exact same steps in the exact same order.
> *   **Agent:** A handyman fixing a leak. They might need to turn off the water, check the pipe, go to the hardware store, come back, and try a different wrench if the first one doesn't fit. The path isn't fixed; it depends on what they find.

*(Diagram: Human → LLM Call → Action → Environment → Feedback back to LLM Call)*

---

**The 5 Workflow Design Patterns**

Here are the five common ways to structure LLM workflows, from simplest to most complex.

**1. Prompt Chaining**

**Concept:** Decompose a task into fixed sub-tasks. The output of one LLM becomes the input for the next.
**Example:** Translating a document.
*   **LLM 1:** Translates English to Spanish.
*   **Gate:** Checks if the translation is valid.
*   **LLM 2:** Polishes the Spanish grammar.
*   **LLM 3:** Formats the text into a final PDF.

*(Diagram: IN → LLM1 → Gate → LLM2 → LLM3 → OUT)*

**2. Routing**

**Concept:** Direct an input into a specialized sub-task, ensuring separation of concerns. One LLM acts as a "traffic cop."
**Example:** Customer Support Desk.
*   **LLM Router:** Reads the customer email.
*   *If it's about billing* → Sends to **Billing LLM**.
*   *If it's a technical issue* → Sends to **Tech Support LLM**.
*   *If it's a refund request* → Sends to **Refund LLM**.

*(Diagram: IN → LLM Router → [LLM1, LLM2, or LLM3] → OUT)*

**3.Parallelization**

**Concept:** Breaking down tasks and running multiple subtasks concurrently to save time, then combining the results.
**Example:** Analyzing a long legal contract.
*   **Coordinator:** Splits the document into three parts.
*   **LLM 1** checks for liability risks.
*   **LLM 2** checks for compliance issues.
*   **LLM 3** summarizes the key dates.
*   **Aggregator:** Combines all three reports into one final summary.

*(Diagram: IN → Coordinator → [LLM1, LLM2, LLM3] → Aggregator → OUT)*

**4. Orchestrator-Worker**

**Concept:** Complex tasks are broken down dynamically. An "Orchestrator" LLM looks at the task, decides what needs to be done, and assigns specific tasks to "Worker" LLMs.
**Example:** Building a simple website.
*   **Orchestrator:** Reads "Build a bakery website." It decides it needs a designer and a coder.
*   **Worker LLM 1:** Writes the HTML structure.
*   **Worker LLM 2:** Writes the CSS styling.
*   **Worker LLM 3:** Writes the text for the homepage.
*   **Synthesizer:** Combines the HTML, CSS, and text into a working site.

*(Diagram: IN → Orchestrator → [LLM1, LLM2, LLM3] → Synthesizer → OUT)*

**5. Evaluator-Optimizer**

**Concept:** One LLM creates content, and another LLM acts as a critic to validate and improve it. This creates a loop until the output is perfect.
**Example:** Writing a marketing email.
*   **LLM Generator:** Writes a draft of the email.
*   **LLM Evaluator:** Reviews the draft against rules (e.g., "Must be under 100 words," "Must include a discount code").
*   *If it fails:* Sends feedback ("Too long, missing code") back to the Generator.
*   *If it passes:* Outputs the final email.

*(Diagram: IN → LLM Generator → Solution → LLM Evaluator → Accepted → OUT. If Rejected, feedback loops back to Generator).*




### Risks of Agentic AI , Guardrails, Evals and Traps to Avoid

**Risks of Agent Frameworks**

*   **Unpredictable path**
*   **Unpredictable output**
*   **Unpredictable costs**

**Monitor (Observability, Evals)**

"Guardrails ensure your agents behave safely, consistently, and within your intended boundaries"


1. **The First Agent Trap**

**User:** "I need an Agent for my business"

**Expert:** "OK - and what business problem is that solving"

**User:** "I want an Strategy Agent to give me advice for my org"

**Expert:** "OK, right, and.. what.. *business problem* is that solving"

2. **The Second Agent Trap Anthropomorphizing**

A common red flag: “Agent Architectures” that assign human-like responsibilities to Agents

**[Diagram of Agent Roles]**
*   Trading Manager Agent
    *   Market Research Agent
    *   Trader Agent
    *   Risk Manager Agent

**The right approach:**
Start simple
Measure performance against a business objective
Add Agents because it improves performance

---

**Takeaways**

*   Build Agent systems to solve problems in a measurable way
*   Decide your Agent Architecture based on performance
*   LLMs generate plausible outputs, not necessarily accurate outputs. Aligning the outputs with business results? That's your job.

---


> Confusing Terminology
>** What is an Agentic Engineer?**
>Someone who uses Agents to Engineer
>or
>someone who engineers Agents?
>**I recommend being explicit:**
>If you've taken my AI Coder course, you are:
>**An engineer skilled at using Coding Agents**
>After this course, you will be:
>**An AI engineer skilled at building Agents**




